# Hands-On Machine Learning — Chapter 2
## End-to-End Machine Learning Project


### Overview

Chapter 2 covers the full workflow for a machine learning project — from understanding the problem to deploying a final system. The chapter emphasizes the importance of clear problem framing, exploratory analysis, robust pipelines, proper evaluation, and reproducibility.

<p align="left"><img src="../fig/figure2.1.png" width="45%"></p>

### Look at the Big Picture

Before coding, clarify the **objective** of the project. Questions to answer:

- What is the **business goal** (reduce churn, detect fraud, predict demand)?
- What kind of **ML task** is it (classification, regression, clustering, etc.)?
- What **performance metric** will define success (precision, recall, RMSE, etc.)?
- What are **constraints** (latency, interpretability, data availability)?

A clear understanding prevents wasted work and ensures alignment with real-world needs.

<p align="left"><img src="../fig/figure2.2.png" width="45%"></p>

### Frame the Problem

Once the big picture is clear, define the **technical framing**:

- **Task type:** Regression predicts continuous values; classification predicts discrete labels.
- **Performance metric:** Align with the cost of errors (precision vs recall, RMSE vs MAE).
- **Success criteria:** Define acceptable thresholds for performance.
- **Assumptions:** Verify data independence (IID), label quality, and time-dependence.

Example: For predicting housing prices, this is a **regression task**, measured by **RMSE**, using historical data of housing features.

<p align="left"><img src="../fig/figure2.3.png" width="45%"></p>

### Get the Data

After defining the task, collect the data. Maintain reproducibility and organization:

- Keep raw data **immutable**; store derived versions separately.
- Track data sources and transformations.
- Use consistent folder structure (`data/raw/`, `data/processed/`, `notebooks/`, `models/`).
- For large datasets, use efficient formats like Parquet or TFRecord.

<p align="left"><img src="../fig/figure2.4.png" width="45%"></p>

### Discover and Visualize the Data

Exploratory Data Analysis (EDA) reveals patterns and potential problems:

- Use `.info()` and `.describe()` to inspect data structure.
- Visualize distributions with histograms and boxplots.
- Create scatterplots and correlation matrices to see relationships.
- Watch for missing values, outliers, or skewed distributions.

<p align="left"><img src="../fig/figure2.5.png" width="45%"></p>

### Create a Test Set

Set aside a test set before analyzing the data. It provides an unbiased final evaluation:

- Use **random** or **stratified sampling** (for class balance).
- For time series, hold out the most recent period.
- Keep the test set **untouched** until the very end.

<p align="left"><img src="../fig/figure2.6.png" width="45%"></p>

### Prepare the Data for Machine Learning Algorithms

Data preparation includes cleaning, transforming, and scaling features:

- **Fix missing values:** with imputation (mean, median, or model-based).
- **Encode categorical features:** using one-hot or ordinal encoding.
- **Feature scaling:** standardize or normalize numerical features.
- **Feature engineering:** create new features that may improve predictions.

These transformations are best encapsulated in **pipelines** for reproducibility.

<p align="left"><img src="../fig/figure2.7.png" width="45%"></p>

### Transformation Pipelines

Use Scikit-Learn's `Pipeline` and `ColumnTransformer` to combine preprocessing and model training:

- Fit preprocessing only on training data (avoids data leakage).
- Apply transformations consistently to training, validation, and test sets.
- Combine numeric and categorical pipelines in a `ColumnTransformer`.

<p align="left"><img src="../fig/figure2.8.png" width="45%"></p>

### Select and Train a Model

Start simple, then progress to more complex models:

- Baseline: Linear Regression, Decision Tree.
- Advanced: Random Forest, Gradient Boosting, Neural Networks.

Use cross-validation to estimate model performance and detect overfitting.

<p align="left"><img src="../fig/figure2.9.png" width="45%"></p>

### Fine-Tune the Model

Once a baseline is established, tune hyperparameters using search strategies:

- **Grid Search:** tests all combinations in a parameter grid.
- **Randomized Search:** samples random parameter combinations, faster for large spaces.
- Evaluate each configuration with cross-validation.

<p align="left"><img src="../fig/figure2.10.png" width="45%"></p>

### Evaluate Your System on the Test Set

After tuning, evaluate the chosen model on the held-out test set. Report RMSE, MAE, R², or other relevant metrics. This gives an unbiased estimate of generalization performance.

<p align="left"><img src="../fig/figure2.11.png" width="45%"></p>

### Launch, Monitor, and Maintain Your System

Deployment involves packaging and serving the model, then monitoring it:

- **Packaging:** use joblib or ONNX to export pipelines.
- **Serving:** expose predictions via REST API or microservice.
- **Monitoring:** watch for data drift and model performance degradation.
- **Retraining:** schedule retraining based on performance or time intervals.

<p align="left"><img src="../fig/figure2.12.png" width="45%"></p>

### Practical Example: Diabetes Regression Pipeline

The following example demonstrates an end-to-end pipeline using Scikit-Learn’s `RandomForestRegressor` on the diabetes dataset:

Steps:
1. Load dataset
2. Split into training/test sets
3. Build preprocessing pipeline
4. Train and evaluate model
5. Perform hyperparameter tuning

<p align="left"><img src="../fig/figure2.13.png" width="45%"></p>

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np

diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = list(X.columns)
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])

model = RandomForestRegressor(random_state=42)
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
print('CV RMSE mean:', -scores.mean())

In [ ]:
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [None, 6, 12]
}
grid_search = GridSearchCV(pipeline, param_grid, cv=4, scoring='neg_root_mean_squared_error', n_jobs=-1)
# grid_search.fit(X_train, y_train)
# print('Best params:', grid_search.best_params_)
# print('Best CV RMSE:', -grid_search.best_score_)

### Notes and Best Practices

- Keep raw data immutable and versioned.
- Use pipelines to prevent data leakage.
- Always evaluate using cross-validation before touching the test set.
- Document parameters, metrics, and data splits.
- Monitor deployed models for drift.

